# ⛏️ PeakyMiner - Tarea 4: Análisis Exploratorio de Datos (EDA)
## Cuaderno 01: Descripción, Estructura Relacional y Calidad de Datos

**Proyecto:** PeakyMiner (`pkminer`)  
**Dataset:** GitHub Agentic Workflows (GH-AW)  
**Publicación Oficial:** [Hugging Face Datasets: apa9s/gh-agentic-workflows](https://huggingface.co/datasets/apa9s/gh-agentic-workflows)  
**Formato de Almacenamiento:** Apache Parquet (Compresión Snappy, Esquema Relacional 3NF)  

---

### 🎯 Objetivos de este Cuaderno
1. **Origen y Carga de Datos:** Cargar de manera reproducible las 3 tablas relacionales del dataset GH-AW con resolución automática de rutas locales y remotas.
2. **Descripción de Tablas y Relaciones:** Caracterizar dimensiones, tipos de datos PyArrow/Pandas, huella en memoria y validar la cardinalidad de las relaciones 1:N entre repositorios, workflows y disparadores.
3. **Auditoría Exhaustiva de Calidad:** Implementar funciones modulares para evaluar valores nulos (opcionales vs. anomalías), duplicidad de registros, unicidad de claves primarias (PK), integridad referencial (FK) y validez sintáctica/contenido de los prompts.
4. **Tratamiento, Limpieza y Exportación:** Justificar analíticamente las decisiones de transformación, aplicar imputaciones controladas y persistir las tablas limpias en `eda/data/processed/` para el análisis del Cuaderno 02.


### 🛠️ Configuración del Entorno y Estilo Visual Uniforme
Configuramos las bibliotecas base (`pandas`, `pyarrow`, `matplotlib`, `seaborn`), verificamos las versiones activas y definimos la plantilla visual uniforme establecida en las directrices técnicas del proyecto.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import seaborn as sns

# Configuración del estilo visual uniforme según especificación
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

# Ajustes de visualización en pandas para tablas claras
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.precision", 3)

print("Versiones de bibliotecas:")
print(f"  - Pandas:     {pd.__version__}")
print(f"  - PyArrow:    {pa.__version__}")
print(f"  - Matplotlib: {plt.matplotlib.__version__}")
print(f"  - Seaborn:    {sns.__version__}")


Versiones de bibliotecas:
  - Pandas:     3.0.5
  - PyArrow:    25.0.1
  - Matplotlib: 3.11.1
  - Seaborn:    0.13.2


---
## 1. Origen y Carga de los Datos

### 1.1 Contexto del Dataset y Enlace Oficial
El dataset analizado recopila la especificación, instrucciones (prompts), metadatos y disparadores de **GitHub Agentic Workflows (GH-AW)**. Estos flujos representan una arquitectura emergente donde los agentes de IA se declaran como documentos Markdown compilados (`.github/workflows/<nombre>.md` junto con su correspondiente `.lock.yml`), ejecutándose sobre la infraestructura de GitHub Actions.

* **Repositorio en Hugging Face:** [apa9s/gh-agentic-workflows](https://huggingface.co/datasets/apa9s/gh-agentic-workflows)
* **Población Minada:** 294 repositorios candidatos de GitHub analizados en dos fases (descubrimiento ligero y extracción profunda de blobs).
* **Licencia:** MIT.

### 1.2 Estrategia de Carga Resiliente
Implementamos un mecanismo de carga con resolución en cascada: busca primero en el directorio local `eda/data/raw/`, con fallback hacia `dataset/` (o rutas relativas al directorio de ejecución del kernel).


In [2]:
def locate_raw_data_dir() -> Path:
    """Localiza de forma robusta el directorio de datos raw."""
    candidates = [
        Path("data/raw"),
        Path("eda/data/raw"),
        Path("../eda/data/raw"),
        Path("../../dataset"),
        Path("../dataset"),
        Path("dataset"),
    ]
    for c in candidates:
        if c.exists() and (c / "repositories.parquet").exists():
            return c.resolve()
    raise FileNotFoundError(
        "No se encontraron los archivos Parquet en las rutas estándar. "
        "Asegúrate de ejecutar desde la raíz del proyecto o haber descargado el dataset."
    )

raw_dir = locate_raw_data_dir()
print(f"📁 Directorio de datos brutos localizado: {raw_dir}")

# Carga de las 3 tablas en memoria
df_repos = pd.read_parquet(raw_dir / "repositories.parquet")
df_workflows = pd.read_parquet(raw_dir / "workflows.parquet")
df_triggers = pd.read_parquet(raw_dir / "workflow_triggers.parquet")

print("✅ Las 3 tablas Parquet se han cargado exitosamente en memoria.")


📁 Directorio de datos brutos localizado: /home/bfl/Code/PeakyMiner/eda/data/raw
✅ Las 3 tablas Parquet se han cargado exitosamente en memoria.


### 1.3 Explicación Semántica de las Entidades del Modelo

El dataset modela el ecosistema GH-AW en tres entidades normalizadas en Tercera Forma Normal (3NF):

1. **`repositories` (`repositories.parquet`):**
   * **Qué representa:** Entidad principal que describe cada repositorio público de GitHub analizado donde se confirmaron flujos de trabajo de agentes autónomos.
   * **Qué modela una fila:** Representa exactamente **un repositorio de software**. Sus atributos capturan popularidad (`stars_count`, `forks_count`), linaje (`is_fork`), gobernanza (`license_spdx`), tecnología (`primary_language`) e identificación canónica (`canonical_url`, `default_branch`, `scanned_at`).
   
2. **`workflows` (`workflows.parquet`):**
   * **Qué representa:** Entidad central del ecosistema agéntico que encapsula la especificación de un flujo de trabajo de agente.
   * **Qué modela una fila:** Representa exactamente **un archivo de definición de agente** (`.github/workflows/*.md`). Sus atributos descomponen el frontmatter YAML (`model_engine`, `tools`, `permissions`) y el cuerpo Markdown (`body_markdown`), registrando además métricas léxicas (`body_word_count`, `body_char_count`) y flags de sanidad (`has_syntax_error`).

3. **`workflow_triggers` (`workflow_triggers.parquet`):**
   * **Qué representa:** Entidad desnormalizada que registra las reglas de activación de los agentes.
   * **Qué modela una fila:** Representa exactamente **un evento de activación** asociado a un workflow (por ejemplo, `push`, `schedule`, `workflow_dispatch`, `issues`). Captura el tipo de evento (`event_type`) y su configuración granular en JSON (`trigger_config`: ramas, cron, tipos de evento).


---
## 2. Descripción de las Tablas y sus Relaciones

### 2.1 Dimensiones, Huella en Memoria y Tipos de Datos
A continuación, computamos el perfil técnico de cada tabla: cantidad de observaciones, número de columnas, volumen en memoria RAM y desglose de tipos de datos.


In [3]:
tables_summary = []

for name, df in [("repositories", df_repos), ("workflows", df_workflows), ("workflow_triggers", df_triggers)]:
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    dtype_counts = df.dtypes.value_counts().to_dict()
    dtype_str = ", ".join([f"{k}: {v}" for k, v in dtype_counts.items()])
    tables_summary.append({
        "Tabla": name,
        "Filas": len(df),
        "Columnas": len(df.columns),
        "Memoria (MB)": round(mem_mb, 2),
        "Distribución Tipos": dtype_str
    })

summary_df = pd.DataFrame(tables_summary)
display(summary_df)


,Tabla,Filas,Columnas,Memoria (MB),Distribución Tipos
0,repositories,292,11,0.05,"str: 7, int64: 2, bool: 1, datetime64[us]: 1"
1,workflows,1368,16,14.51,"str: 12, int64: 2, object: 1, bool: 1"
2,workflow_triggers,3475,4,0.76,str: 4


### 2.2 Muestra Inicial de Cada Tabla
Inspeccionamos visualmente una muestra representativa de cada entidad para constatar su estructura columnar.


In [4]:
print("=== Muestra de Repositorios (Top 3) ===")
display(df_repos[["repo_id", "owner", "name", "stars_count", "primary_language", "license_spdx"]].head(3))

print("\n=== Muestra de Workflows (Top 3) ===")
display(df_workflows[["workflow_id", "repo_id", "basename", "model_engine", "body_word_count", "has_syntax_error"]].head(3))

print("\n=== Muestra de Triggers (Top 5) ===")
display(df_triggers[["trigger_id", "workflow_id", "event_type", "trigger_config"]].head(5))


=== Muestra de Repositorios (Top 3) ===


,repo_id,owner,name,stars_count,primary_language,license_spdx
0,b7efa475-9df5-5d88-9f43-13f3feabf499,mcpc-tech,dev-inspector-mcp,47,TypeScript,MIT
1,80f9c9da-4b70-5d63-aae6-0a4c06d026b9,benbalter,wordpress-static-site-exporter,1078,PHP,GPL-3.0
2,dae5bb85-3681-5fc2-8b72-33b81de1e3a0,prasadhonrao,ghcp-cli-cheatsheet,11,TypeScript,MIT



=== Muestra de Workflows (Top 3) ===


,workflow_id,repo_id,basename,model_engine,body_word_count,has_syntax_error
0,101c8404-8271-5a33-96de-8b6e04201ee9,b7efa475-9df5-5d88-9f43-13f3feabf499,daily-repo-status,NaN,105,False
1,e9ddd950-9ff6-57f3-aaeb-1bfce7b0d8fa,80f9c9da-4b70-5d63-aae6-0a4c06d026b9,issue-triage,NaN,298,False
2,69b13c38-870d-53cb-90d5-3828db09bf45,80f9c9da-4b70-5d63-aae6-0a4c06d026b9,pr-review,NaN,287,False



=== Muestra de Triggers (Top 5) ===


,trigger_id,workflow_id,event_type,trigger_config
0,a6d230c5-0c83-58af-bb1d-8f7e9399a608,101c8404-8271-5a33-96de-8b6e04201ee9,schedule,"""daily"""
1,079b156d-0ff5-5be2-a3b6-79185736cb9d,101c8404-8271-5a33-96de-8b6e04201ee9,workflow_dispatch,{}
2,2a6c46cc-ad96-5c6f-ae92-7cc293f1dee1,e9ddd950-9ff6-57f3-aaeb-1bfce7b0d8fa,issues,"{""types"": [""opened""]}"
3,b16a28d1-2e24-5fc0-97be-7297f4f3f099,69b13c38-870d-53cb-90d5-3828db09bf45,pull_request,"{""types"": [""opened"", ""synchronize""]}"
4,fe48e1a0-7cad-55e3-8cd1-1fec805caea5,0ac746c8-b066-5992-ac81-6cf17da9e208,release,"{""types"": [""created""]}"


### 2.3 Identificación de Claves Primarias y Foráneas (Diagrama Relacional)

La arquitectura de datos se fundamenta en identificadores deterministas calculados mediante el estándar **UUIDv5** (RFC 4122) sobre espacios de nombres canónicos:

```
┌──────────────────────────────────────┐
│             repositories             │
├──────────────────────────────────────┤
│ PK: repo_id (UUIDv5)                 │◄─────────┐
│     owner, name, stars_count...      │          │
└──────────────────┬───────────────────┘          │
                   │                              │ 1:N
                   │ 1:N                          │ (Un repositorio aloja N workflows)
                   ▼                              │
┌──────────────────────────────────────┐          │
│              workflows               │          │
├──────────────────────────────────────┤          │
│ PK: workflow_id (UUIDv5)             │◄──┐      │
│ FK: repo_id (UUIDv5) ────────────────┼───┘──────┘
│     file_path, model_engine, body... │   │
└──────────────────┬───────────────────┘   │
                   │                       │ 1:N
                   │ 1:N                   │ (Un workflow responde a N triggers)
                   ▼                       │
┌──────────────────────────────────────┐   │
│          workflow_triggers           │   │
├──────────────────────────────────────┤   │
│ PK: trigger_id (UUIDv5)              │   │
│ FK: workflow_id (UUIDv5) ────────────┼───┘
│     event_type, trigger_config       │
└──────────────────────────────────────┘
```

* **Relación 1: `repositories.repo_id` (PK) $\leftrightarrow$ `workflows.repo_id` (FK) [Cardinalidad 1:N]:**
  Un único repositorio puede albergar desde 1 hasta decenas de definiciones de agentes en `.github/workflows/`. Cada workflow pertenece estrictamente a un único repositorio padre.
* **Relación 2: `workflows.workflow_id` (PK) $\leftrightarrow$ `workflow_triggers.workflow_id` (FK) [Cardinalidad 1:N]:**
  Un agente individual puede responder a múltiples eventos de disparo (por ejemplo, ejecutarse periódicamente con `schedule`, responder manualmente con `workflow_dispatch` y reaccionar a un `pull_request`). Cada evento está asociado a un único workflow.


### 2.4 Análisis Comparativo: Filas Totales vs. Identificadores Únicos
Calculamos formalmente la unicidad de las claves y determinamos cuántos repositorios de la muestra poseen al menos un flujo extraído.


In [5]:
n_repos_total = len(df_repos)
n_repos_unique = df_repos["repo_id"].nunique()
n_repos_in_wf = df_workflows["repo_id"].nunique()

n_wf_total = len(df_workflows)
n_wf_unique = df_workflows["workflow_id"].nunique()
n_wf_in_trig = df_triggers["workflow_id"].nunique()

n_trig_total = len(df_triggers)
n_trig_unique = df_triggers["trigger_id"].nunique()

relational_stats = pd.DataFrame([
    {
        "Entidad": "repositories",
        "Filas Totales": n_repos_total,
        "IDs Únicos": n_repos_unique,
        "¿Todas las PKs son únicas?": n_repos_total == n_repos_unique,
        "Referencias en Tabla Hija": f"{n_repos_in_wf} repos tienen workflows ({n_repos_in_wf / n_repos_unique:.1%})"
    },
    {
        "Entidad": "workflows",
        "Filas Totales": n_wf_total,
        "IDs Únicos": n_wf_unique,
        "¿Todas las PKs son únicas?": n_wf_total == n_wf_unique,
        "Referencias en Tabla Hija": f"{n_wf_in_trig} workflows tienen triggers ({n_wf_in_trig / n_wf_unique:.1%})"
    },
    {
        "Entidad": "workflow_triggers",
        "Filas Totales": n_trig_total,
        "IDs Únicos": n_trig_unique,
        "¿Todas las PKs son únicas?": n_trig_total == n_trig_unique,
        "Referencias en Tabla Hija": "N/A (Tabla hoja del modelo)"
    },
])

display(relational_stats)
print("📊 Ratios Relacionales:")
print(f"  - Promedio de Workflows por Repositorio: {n_wf_total / n_repos_unique:.2f}")
print(f"  - Promedio de Triggers por Workflow:     {n_trig_total / n_wf_unique:.2f}")


,Entidad,Filas Totales,IDs Únicos,¿Todas las PKs son únicas?,Referencias en Tabla Hija
0,repositories,292,292,True,292 repos tienen workflows (100.0%)
1,workflows,1368,1368,True,1367 workflows tienen triggers (99.9%)
2,workflow_triggers,3475,3475,True,N/A (Tabla hoja del modelo)


📊 Ratios Relacionales:
  - Promedio de Workflows por Repositorio: 4.68
  - Promedio de Triggers por Workflow:     2.54


---
## 3. Revisión Exhaustiva de Calidad de Datos

Diseñamos e implementamos una batería de funciones modulares de auditoría para verificar:
1. Valores nulos (clasificados por relevancia semántica).
2. Duplicados exactos en filas completas.
3. Integridad y unicidad de las Claves Primarias (PK).
4. Integridad Referencial (detección de registros huérfanos entre PKs y FKs).
5. Consistencia sintáctica y existencia de contenido en los prompts.


In [6]:
# ==============================================================================
# FUNCIONES MODULARES DE AUDITORÍA DE CALIDAD
# ==============================================================================

def audit_null_values(df: pd.DataFrame, table_name: str, optional_fields: set[str]) -> pd.DataFrame:
    """Calcula valores nulos por columna y clasifica según regla de negocio."""
    null_counts = df.isnull().sum()
    total_rows = len(df)
    records = []
    for col, count in null_counts.items():
        pct = (count / total_rows) * 100
        is_optional = col in optional_fields
        if count == 0:
            classification = "Completo (0 nulos)"
        elif is_optional:
            classification = "Opcional Permitido (Negocio)"
        else:
            classification = "⚠️ Anomalía Inesperada"
        records.append({
            "Tabla": table_name,
            "Columna": col,
            "Nulos (Conteo)": count,
            "Nulos (%)": round(pct, 2),
            "Clasificación": classification
        })
    return pd.DataFrame(records)

def audit_duplicates(df: pd.DataFrame, table_name: str) -> dict:
    """Verifica duplicidad de filas completas en la tabla manejando columnas no hasheables."""
    df_check = df.copy()
    for col in df_check.columns:
        if df_check[col].apply(lambda x: isinstance(x, (list, np.ndarray))).any():
            df_check[col] = df_check[col].apply(lambda x: tuple(x) if isinstance(x, (list, np.ndarray)) else x)
    dup_count = df_check.duplicated().sum()
    return {
        "Tabla": table_name,
        "Total Filas": len(df),
        "Filas Duplicadas Exactas": dup_count,
        "Estado": "✅ Sin duplicados" if dup_count == 0 else f"⚠️ {dup_count} duplicados detectados"
    }

def audit_primary_key(df: pd.DataFrame, pk_col: str, table_name: str) -> dict:
    """Verifica que la clave primaria no tenga nulos y sea estrictamente única."""
    null_pks = df[pk_col].isnull().sum()
    unique_pks = df[pk_col].nunique()
    total_rows = len(df)
    is_valid = (null_pks == 0) and (unique_pks == total_rows)
    return {
        "Tabla": table_name,
        "PK": pk_col,
        "Nulos en PK": null_pks,
        "Valores Únicos": unique_pks,
        "Filas Totales": total_rows,
        "Integridad PK": "✅ Válida y Única" if is_valid else "❌ Violación de PK"
    }

def audit_referential_integrity(
    parent_df: pd.DataFrame, parent_pk: str,
    child_df: pd.DataFrame, child_fk: str,
    relationship_label: str
) -> dict:
    """Verifica la ausencia de registros huérfanos en una relación 1:N."""
    valid_parents = set(parent_df[parent_pk].dropna().unique())
    orphans_mask = ~child_df[child_fk].isin(valid_parents)
    orphan_count = orphans_mask.sum()
    return {
        "Relación": relationship_label,
        "Total en Tabla Hija": len(child_df),
        "Registros Huérfanos": orphan_count,
        "Estado": "✅ Integridad Perfecta (0 huérfanos)" if orphan_count == 0 else f"❌ {orphan_count} huérfanos"
    }

print("✅ Funciones de auditoría de calidad definidas correctamente.")


✅ Funciones de auditoría de calidad definidas correctamente.


### 3.1 Auditoría 1: Diagnóstico de Valores Nulos
Examinamos la presencia de nulos en las 3 tablas, contrastando si corresponden a campos opcionales del estándar GH-AW o a inconsistencias técnicas.


In [7]:
# Definición de campos opcionales por regla de diseño
optional_repo = {"primary_language", "license_spdx"}
optional_wf = {"name", "description", "model_engine", "tools", "permissions"}
optional_trig = set()

null_repos_report = audit_null_values(df_repos, "repositories", optional_repo)
null_wf_report = audit_null_values(df_workflows, "workflows", optional_wf)
null_trig_report = audit_null_values(df_triggers, "workflow_triggers", optional_trig)

all_nulls = pd.concat([null_repos_report, null_wf_report, null_trig_report], ignore_index=True)
display(all_nulls[all_nulls["Nulos (Conteo)"] > 0])


,Tabla,Columna,Nulos (Conteo),Nulos (%),Clasificación
6,repositories,primary_language,1,0.34,Opcional Permitido (Negocio)
8,repositories,license_spdx,29,9.93,Opcional Permitido (Negocio)
17,workflows,name,665,48.61,Opcional Permitido (Negocio)
18,workflows,description,133,9.72,Opcional Permitido (Negocio)
19,workflows,model_engine,511,37.35,Opcional Permitido (Negocio)
20,workflows,tools,118,8.63,Opcional Permitido (Negocio)
21,workflows,permissions,16,1.17,Opcional Permitido (Negocio)


#### Interpretación del Perfil de Nulos:
* **En `repositories`:**
  * `license_spdx` (29 nulos, 9.9%): Es común en proyectos de código abierto donde no se ha formalizado un archivo LICENSE o no se reconoce un identificador SPDX estándar.
  * `primary_language` (1 nulo, 0.3%): Corresponde a un repositorio con contenido exclusivo de documentación o scripts misceláneos sin lenguaje clasificado por GitHub Linguist.
* **En `workflows`:**
  * `name` (665 nulos, 48.6%): La especificación GH-AW permite omitir la clave `name` en el frontmatter; en dicho caso, el sistema infiere el identificador a partir del nombre del archivo (`basename`).
  * `description` (133 nulos, 9.7%): Descripción contextual opcional del agente.
  * `model_engine` (511 nulos, 37.4%): La omisión de este campo es legal en GH-AW: el agente asume el modelo asignado por defecto en el runtime de GitHub Copilot.
  * `tools` (118 nulos, 8.6%): Agentes que no declaran herramientas externas o usan las herramientas base del entorno.
  * `permissions` (16 nulos, 1.2%): Permisos estándar de GitHub Actions heredados del repositorio.
* **En `workflow_triggers`:**
  * **0 nulos en todas las columnas.** Integridad perfecta.


### 3.2 Auditoría 2: Duplicados Exactos
Comprobamos si existen filas duplicadas completas en cualquiera de las entidades.


In [8]:
dups_report = pd.DataFrame([
    audit_duplicates(df_repos, "repositories"),
    audit_duplicates(df_workflows, "workflows"),
    audit_duplicates(df_triggers, "workflow_triggers"),
])
display(dups_report)


,Tabla,Total Filas,Filas Duplicadas Exactas,Estado
0,repositories,292,0,✅ Sin duplicados
1,workflows,1368,0,✅ Sin duplicados
2,workflow_triggers,3475,0,✅ Sin duplicados


### 3.3 Auditoría 3: Integridad de Claves Primarias (PKs)
Comprobamos que las claves primarias no contengan valores nulos y sean 100% únicas.


In [9]:
pk_report = pd.DataFrame([
    audit_primary_key(df_repos, "repo_id", "repositories"),
    audit_primary_key(df_workflows, "workflow_id", "workflows"),
    audit_primary_key(df_triggers, "trigger_id", "workflow_triggers"),
])
display(pk_report)


,Tabla,PK,Nulos en PK,Valores Únicos,Filas Totales,Integridad PK
0,repositories,repo_id,0,292,292,✅ Válida y Única
1,workflows,workflow_id,0,1368,1368,✅ Válida y Única
2,workflow_triggers,trigger_id,0,3475,3475,✅ Válida y Única


### 3.4 Auditoría 4: Integridad Referencial (Detección de Huérfanos)
Verificamos que cada clave foránea (`FK`) apunte a una clave primaria (`PK`) existente en la tabla padre.


In [10]:
fk_report = pd.DataFrame([
    audit_referential_integrity(
        df_repos, "repo_id",
        df_workflows, "repo_id",
        "repositories (1) -> workflows (N)"
    ),
    audit_referential_integrity(
        df_workflows, "workflow_id",
        df_triggers, "workflow_id",
        "workflows (1) -> workflow_triggers (N)"
    ),
])
display(fk_report)


,Relación,Total en Tabla Hija,Registros Huérfanos,Estado
0,repositories (1) -> workflows (N),1368,0,✅ Integridad Perfecta (0 huérfanos)
1,workflows (1) -> workflow_triggers (N),3475,0,✅ Integridad Perfecta (0 huérfanos)


### 3.5 Auditoría 5: Consistencia Sintáctica y Prompts Vacíos
Auditamos si existen flujos de trabajo marcados con errores sintácticos en el frontmatter (`has_syntax_error == True`) o con cuerpos vacíos (`body_char_count == 0` / `body_word_count == 0`).


In [11]:
syntax_errors_count = df_workflows["has_syntax_error"].sum()
empty_body_chars = (df_workflows["body_char_count"] == 0).sum()
empty_body_words = (df_workflows["body_word_count"] == 0).sum()

syntax_report = pd.DataFrame([
    {
        "Métrica de Contenido": "Workflows con error sintáctico YAML (has_syntax_error == True)",
        "Cantidad": syntax_errors_count,
        "Porcentaje": f"{(syntax_errors_count / len(df_workflows)):.2%}",
        "Diagnóstico": "✅ Todos los frontmatters parsearon correctamente" if syntax_errors_count == 0 else "⚠️ Revisar YAMLs corruptos"
    },
    {
        "Métrica de Contenido": "Workflows con body vacío (body_char_count == 0)",
        "Cantidad": empty_body_chars,
        "Porcentaje": f"{(empty_body_chars / len(df_workflows)):.2%}",
        "Diagnóstico": "✅ Todos los workflows contienen directivas/prompts" if empty_body_chars == 0 else "⚠️ Existen agentes sin prompt"
    },
    {
        "Métrica de Contenido": "Workflows con 0 palabras (body_word_count == 0)",
        "Cantidad": empty_body_words,
        "Porcentaje": f"{(empty_body_words / len(df_workflows)):.2%}",
        "Diagnóstico": "✅ Todos los workflows tienen palabras analizables" if empty_body_words == 0 else "⚠️ Existen workflows sin palabras"
    }
])

display(syntax_report)


,Métrica de Contenido,Cantidad,Porcentaje,Diagnóstico
0,Workflows con error sintáctico YAML (has_syntax_error ==...,0,0.00%,✅ Todos los frontmatters parsearon correctamente
1,Workflows con body vacío (body_char_count == 0),0,0.00%,✅ Todos los workflows contienen directivas/prompts
2,Workflows con 0 palabras (body_word_count == 0),0,0.00%,✅ Todos los workflows tienen palabras analizables


---
## 4. Tratamiento de los Problemas Encontrados y Exportación

### 4.1 Discusión y Justificación Metodológica de Decisiones

1. **Conservación Total de Registros (Sin Exclusiones de Filas):**
   * Como se evidenció en la auditoría, **no existen claves duplicadas ni registros huérfanos** en ninguna de las tres tablas.
   * La totalidad de los 292 repositorios confirmados y 1,368 workflows son íntegros, por lo que **no se descarta ninguna fila** del dataset original, garantizando representatividad plena.

2. **Imputación de Herramientas Vacías (`tools`):**
   * En `workflows.parquet`, la columna `tools` es de tipo lista (`list<element: string>`). Actualmente contiene 118 valores nulos (`NaN` en pandas).
   * **Decisión:** Imputar estos nulos con una lista vacía `[]`. Esto permite aplicar operaciones de listas (e.g., `len(x)` o `.explode()`) sin errores de ejecución ni necesidad de comprobaciones defensivas repetitivas en el análisis exploratorio.

3. **Estandarización de Nulos en Metadatos Opcionales:**
   * En `license_spdx` y `primary_language`, conservamos los nulos explícitos (y creamos columnas auxiliares imputadas con `"Unknown"` / `"None"` según sea requerido en categorización), ya que representan la ausencia real del atributo en GitHub.
   * En `model_engine`, preservamos el valor original para permitir evaluar explícitamente el impacto del "motor por defecto" frente a modelos declarados explícitamente.

4. **Preservación de Tipos PyArrow:**
   * Al exportar a `eda/data/processed/`, se preservan las estructuras relacionales con tipos de datos nativos compatibles con Apache Parquet y compresión Snappy.


In [12]:
# Copias explícitas para el pipeline de procesamiento
df_repos_clean = df_repos.copy()
df_workflows_clean = df_workflows.copy()
df_triggers_clean = df_triggers.copy()

# Transformación 1: Imputación controlada de tools nulos a listas vacías
null_tools_before = df_workflows_clean["tools"].isnull().sum()
df_workflows_clean["tools"] = df_workflows_clean["tools"].apply(
    lambda x: [] if (x is None or (isinstance(x, float) and pd.isna(x))) else list(x)
)
null_tools_after = df_workflows_clean["tools"].isnull().sum()

# Transformación 2: Estandarización de string en campos opcionales para evitar inconsistencias
df_repos_clean["primary_language_imputed"] = df_repos_clean["primary_language"].fillna("Not Specified")
df_repos_clean["license_spdx_imputed"] = df_repos_clean["license_spdx"].fillna("Unlicensed/Proprietary")

transformation_summary = pd.DataFrame([
    {
        "Tabla": "workflows",
        "Operación": "Imputación de tools nulos -> []",
        "Registros Modificados": null_tools_before,
        "Nulos Resultantes": null_tools_after
    },
    {
        "Tabla": "repositories",
        "Operación": "Columnas imputadas auxiliares (lenguaje, licencia)",
        "Registros Modificados": len(df_repos_clean),
        "Nulos Resultantes": 0
    },
    {
        "Tabla": "workflow_triggers",
        "Operación": "Validación de esquema y tipos",
        "Registros Modificados": 0,
        "Nulos Resultantes": 0
    }
])

display(transformation_summary)


,Tabla,Operación,Registros Modificados,Nulos Resultantes
0,workflows,Imputación de tools nulos -> [],118,0
1,repositories,"Columnas imputadas auxiliares (lenguaje, licencia)",292,0
2,workflow_triggers,Validación de esquema y tipos,0,0


### 4.2 Exportación de Tablas Preparadas a `eda/data/processed/`
Persistimos las versiones limpias y tipadas de las tres tablas en formato Parquet con compresión Snappy en el directorio `eda/data/processed/`.


In [13]:
# Definición del directorio de destino
def locate_processed_dir() -> Path:
    candidates = [
        Path("data/processed"),
        Path("eda/data/processed"),
        Path("../eda/data/processed"),
    ]
    for c in candidates:
        if c.parent.exists():
            c.mkdir(parents=True, exist_ok=True)
            return c.resolve()
    p = Path("eda/data/processed").resolve()
    p.mkdir(parents=True, exist_ok=True)
    return p

processed_dir = locate_processed_dir()
print(f"📁 Directorio de salida para datos procesados: {processed_dir}")

# Exportación a Parquet con PyArrow
repo_clean_path = processed_dir / "repositories_clean.parquet"
wf_clean_path = processed_dir / "workflows_clean.parquet"
trig_clean_path = processed_dir / "workflow_triggers_clean.parquet"

df_repos_clean.to_parquet(repo_clean_path, index=False, engine="pyarrow", compression="snappy")
df_workflows_clean.to_parquet(wf_clean_path, index=False, engine="pyarrow", compression="snappy")
df_triggers_clean.to_parquet(trig_clean_path, index=False, engine="pyarrow", compression="snappy")

print("✅ Tablas limpias exportadas con éxito:")
for p in [repo_clean_path, wf_clean_path, trig_clean_path]:
    size_kb = p.stat().st_size / 1024
    print(f"  - {p.name}: {size_kb:.1f} KB")


📁 Directorio de salida para datos procesados: /home/bfl/Code/PeakyMiner/eda/data/processed
✅ Tablas limpias exportadas con éxito:
  - repositories_clean.parquet: 35.1 KB
  - workflows_clean.parquet: 6459.0 KB
  - workflow_triggers_clean.parquet: 292.7 KB


### 4.3 Verificación Final de Integridad de los Archivos Procesados
Leemos nuevamente los archivos generados para asegurar que el Cuaderno 02 podrá consumirlos sin ninguna dependencia de memoria del Cuaderno 01.


In [14]:
# Comprobación de lectura independiente
test_repos = pd.read_parquet(repo_clean_path)
test_wf = pd.read_parquet(wf_clean_path)
test_trig = pd.read_parquet(trig_clean_path)

verification_report = pd.DataFrame([
    {"Archivo Procesado": "repositories_clean.parquet", "Filas Verificadas": len(test_repos), "Columnas": len(test_repos.columns)},
    {"Archivo Procesado": "workflows_clean.parquet", "Filas Verificadas": len(test_wf), "Columnas": len(test_wf.columns)},
    {"Archivo Procesado": "workflow_triggers_clean.parquet", "Filas Verificadas": len(test_trig), "Columnas": len(test_trig.columns)},
])

display(verification_report)
print("🎉 Cuaderno 01 finalizado exitosamente. Los datos están listos para la exploración en el Cuaderno 02.")


,Archivo Procesado,Filas Verificadas,Columnas
0,repositories_clean.parquet,292,13
1,workflows_clean.parquet,1368,16
2,workflow_triggers_clean.parquet,3475,4


🎉 Cuaderno 01 finalizado exitosamente. Los datos están listos para la exploración en el Cuaderno 02.
